What is Spark?
Apache Spark is a unified analytics engine for large-scale data processing. It provides high-level APIs in Python, SQL, Scala, and Java, and supports general computation graphs for data analysis. Spark is designed for speed, ease of use, and sophisticated analytics.

Spark Architecture
Core Components:

Driver Program: Runs the main() function, creates SparkContext, and coordinates execution
Cluster Manager: Allocates resources (YARN, Kubernetes, Mesos, or Standalone)
Executors: Worker nodes that run tasks and store data in memory/disk
Tasks: Units of work sent to executors
Key Concept - RDDs (Resilient Distributed Datasets): Immutable distributed collections of objects that can be processed in parallel with fault tolerance through lineage tracking.

Spark vs MapReduce
Aspect
Spark
MapReduce
Speed
10-100x faster (in-memory)
Slower (disk-based)
Processing
Batch, streaming, ML, graph
Batch only
Ease of Use
High-level APIs, interactive
More verbose, no interactivity
Data Sharing
In-memory (RAM)
Disk between stages
Fault Tolerance
Lineage-based recomputation
Replication
Hadoop vs Spark
Hadoop is an ecosystem (HDFS + MapReduce + YARN), while Spark is a processing engine that can run on Hadoop or standalone. Spark can use HDFS for storage but replaces MapReduce with a faster execution engine.

Computation Model
MapReduce: Two-stage (Map → Shuffle → Reduce), writes intermediate results to disk

Spark: Directed Acyclic Graph (DAG) of operations with:

Transformations (lazy): map, filter, groupBy - build execution plan
Actions (eager): collect, count, save - trigger computation
In-memory processing: Data cached between operations for speed
Lazy evaluation: Optimizes execution plan before running
Spark's DAG scheduler optimizes the entire pipeline, enabling multi-stage operations without disk I/O between stages.

## What is RDD?

**RDD (Resilient Distributed Dataset)** is the fundamental data structure of Apache Spark. It's an immutable, distributed collection of objects that can be processed in parallel across a cluster.

## Key Characteristics

* **Resilient**: Fault-tolerant through lineage tracking - if a partition is lost, Spark can recompute it from the original data source
* **Distributed**: Data is partitioned across multiple nodes in the cluster
* **Immutable**: Once created, RDDs cannot be changed - transformations create new RDDs
* **Lazy Evaluation**: Transformations are not executed until an action is called
* **In-Memory**: Can cache data in memory for faster iterative operations

## Creating RDDs

**Three ways to create RDDs:**

1. **Parallelizing an existing collection**:
   ```python
   data = [1, 2, 3, 4, 5]
   rdd = sc.parallelize(data)
   ```

2. **Loading external datasets**:
   ```python
   rdd = sc.textFile("hdfs://path/to/file.txt")
   ```

3. **Transforming existing RDDs**:
   ```python
   new_rdd = rdd.map(lambda x: x * 2)
   ```

## RDD Operations

### Transformations (Lazy)
Create new RDDs from existing ones:

* `map(func)` - Apply function to each element
* `filter(func)` - Select elements matching predicate
* `flatMap(func)` - Map and flatten results
* `reduceByKey(func)` - Aggregate by key
* `groupByKey()` - Group values by key
* `join()` - Join two RDDs
* `distinct()` - Remove duplicates
* `union()` - Combine two RDDs

### Actions (Eager)
Trigger computation and return results:

* `collect()` - Return all elements to driver
* `count()` - Count elements
* `first()` - Return first element
* `take(n)` - Return first n elements
* `reduce(func)` - Aggregate elements
* `saveAsTextFile(path)` - Save to file
* `foreach(func)` - Apply function to each element

## Lineage and Fault Tolerance

RDDs track their **lineage** (sequence of transformations from source data). If a partition fails:
* Spark uses the lineage to recompute only the lost partition
* No data replication needed
* Automatic recovery without user intervention

## Example

```python
# Create RDD from list
rdd = sc.parallelize([1, 2, 3, 4, 5])

# Transformations (lazy - not executed yet)
squared_rdd = rdd.map(lambda x: x ** 2)
filtered_rdd = squared_rdd.filter(lambda x: x > 10)

# Action (triggers execution)
result = filtered_rdd.collect()  # [16, 25]
```

## RDD vs DataFrame/Dataset

While RDDs are the foundation, Spark now recommends using **DataFrames** and **Datasets** for most use cases:

| Feature | RDD | DataFrame/Dataset |
| --- | --- | --- |
| API Level | Low-level | High-level |
| Optimization | Manual | Catalyst optimizer |
| Schema | No schema | Structured schema |
| Performance | Slower | Faster (Tungsten execution) |
| Use Case | Complex custom logic | Structured data processing |

**When to use RDDs:**
* Need fine-grained control over data
* Working with unstructured data
* Custom partitioning logic
* Legacy code maintenance

Quick Summary
RDD (Resilient Distributed Dataset) is Spark's fundamental data structure - an immutable, distributed collection of objects processed in parallel across a cluster.

Core Features:

Resilient - Fault-tolerant via lineage tracking (recomputes lost partitions automatically)
Distributed - Partitioned across cluster nodes
Immutable - Transformations create new RDDs
Lazy - Computations deferred until actions are called
In-Memory - Can cache data for iterative operations
Two Operation Types:

Transformations (lazy): map, filter, reduceByKey - build execution plan
Actions (eager): collect, count, reduce - trigger computation
Modern Context: While RDDs are Spark's foundation, DataFrames and Datasets are now recommended for most use cases due to Catalyst optimization and better performance. Use RDDs when you need fine-grained control, work with unstructured data, or require custom partitioning logic.

## RDD Operations - Complete List

RDD operations are divided into two categories: **Transformations** (lazy) and **Actions** (eager).

---

## 1. TRANSFORMATIONS (Lazy Operations)

Transformations create new RDDs from existing ones. They are not executed immediately - Spark builds a DAG and executes only when an action is called.

### A. Narrow Transformations (No Shuffle)
Data from each partition goes to at most one partition of the output RDD.

#### `map(func)`
**Use**: Apply a function to each element and return a new RDD
```python
rdd = sc.parallelize([1, 2, 3, 4])
mapped = rdd.map(lambda x: x * 2)  # [2, 4, 6, 8]
```

#### `filter(func)`
**Use**: Select elements that satisfy a condition
```python
filtered = rdd.filter(lambda x: x > 2)  # [3, 4]
```

#### `flatMap(func)`
**Use**: Apply function and flatten results (one-to-many mapping)
```python
rdd = sc.parallelize(["hello world", "hi there"])
words = rdd.flatMap(lambda x: x.split(" "))  # ["hello", "world", "hi", "there"]
```

#### `mapPartitions(func)`
**Use**: Apply function to each partition (more efficient than map for expensive operations)
```python
def process_partition(iterator):
    # Setup expensive operation once per partition
    for x in iterator:
        yield x * 2
        
result = rdd.mapPartitions(process_partition)
```

#### `mapPartitionsWithIndex(func)`
**Use**: Like mapPartitions but includes partition index
```python
def process_with_index(index, iterator):
    return [(index, x) for x in iterator]
    
result = rdd.mapPartitionsWithIndex(process_with_index)
```

#### `sample(withReplacement, fraction, seed)`
**Use**: Sample a fraction of data with or without replacement
```python
sampled = rdd.sample(False, 0.5, seed=42)  # 50% sample without replacement
```

#### `union(otherRDD)`
**Use**: Combine two RDDs (simple concatenation)
```python
rdd1 = sc.parallelize([1, 2, 3])
rdd2 = sc.parallelize([4, 5, 6])
combined = rdd1.union(rdd2)  # [1, 2, 3, 4, 5, 6]
```

#### `distinct([numPartitions])`
**Use**: Remove duplicate elements
```python
rdd = sc.parallelize([1, 2, 2, 3, 3, 3])
unique = rdd.distinct()  # [1, 2, 3]
```

#### `glom()`
**Use**: Convert each partition into a list (useful for debugging)
```python
rdd = sc.parallelize([1, 2, 3, 4], 2)
glommed = rdd.glom().collect()  # [[1, 2], [3, 4]]
```

---

### B. Wide Transformations (Require Shuffle)
Data from multiple partitions may need to be combined.

#### `groupByKey()`
**Use**: Group values by key (for key-value pairs)
```python
rdd = sc.parallelize([("a", 1), ("b", 2), ("a", 3)])
grouped = rdd.groupByKey()  # [("a", [1, 3]), ("b", [2])]
```
**⚠️ Warning**: Avoid for aggregations - use reduceByKey instead (more efficient)

#### `reduceByKey(func, [numPartitions])`
**Use**: Aggregate values by key using a reduce function (more efficient than groupByKey)
```python
rdd = sc.parallelize([("a", 1), ("b", 2), ("a", 3)])
summed = rdd.reduceByKey(lambda x, y: x + y)  # [("a", 4), ("b", 2)]
```

#### `aggregateByKey(zeroValue, seqFunc, combFunc)`
**Use**: Aggregate values by key with different types for input and output
```python
# Calculate sum and count for each key
rdd = sc.parallelize([("a", 1), ("a", 2), ("b", 3)])
result = rdd.aggregateByKey(
    (0, 0),  # (sum, count)
    lambda acc, value: (acc[0] + value, acc[1] + 1),
    lambda acc1, acc2: (acc1[0] + acc2[0], acc1[1] + acc2[1])
)
```

#### `sortByKey([ascending], [numPartitions])`
**Use**: Sort RDD by keys
```python
rdd = sc.parallelize([(3, "c"), (1, "a"), (2, "b")])
sorted_rdd = rdd.sortByKey()  # [(1, "a"), (2, "b"), (3, "c")]
```

#### `join(otherRDD, [numPartitions])`
**Use**: Inner join two RDDs by key
```python
rdd1 = sc.parallelize([("a", 1), ("b", 2)])
rdd2 = sc.parallelize([("a", 3), ("c", 4)])
joined = rdd1.join(rdd2)  # [("a", (1, 3))]
```

#### `leftOuterJoin(otherRDD)`
**Use**: Left outer join - keeps all keys from left RDD
```python
result = rdd1.leftOuterJoin(rdd2)  # [("a", (1, 3)), ("b", (2, None))]
```

#### `rightOuterJoin(otherRDD)`
**Use**: Right outer join - keeps all keys from right RDD
```python
result = rdd1.rightOuterJoin(rdd2)  # [("a", (1, 3)), ("c", (None, 4))]
```

#### `fullOuterJoin(otherRDD)`
**Use**: Full outer join - keeps all keys from both RDDs
```python
result = rdd1.fullOuterJoin(rdd2)  # [("a", (1, 3)), ("b", (2, None)), ("c", (None, 4))]
```

#### `cogroup(otherRDD)`
**Use**: Group data from both RDDs sharing the same key
```python
result = rdd1.cogroup(rdd2)  # [("a", ([1], [3])), ("b", ([2], [])), ("c", ([], [4]))]
```

#### `cartesian(otherRDD)`
**Use**: Create Cartesian product (all pairs from both RDDs)
```python
rdd1 = sc.parallelize([1, 2])
rdd2 = sc.parallelize(["a", "b"])
result = rdd1.cartesian(rdd2)  # [(1, "a"), (1, "b"), (2, "a"), (2, "b")]
```
**⚠️ Warning**: Very expensive - output size = m × n

#### `repartition(numPartitions)`
**Use**: Change number of partitions (causes full shuffle)
```python
repartitioned = rdd.repartition(10)  # Increase to 10 partitions
```

#### `coalesce(numPartitions)`
**Use**: Decrease number of partitions efficiently (no shuffle if reducing)
```python
coalesced = rdd.coalesce(2)  # Reduce to 2 partitions
```

#### `partitionBy(numPartitions, [partitionFunc])`
**Use**: Partition data using custom partitioner (for key-value RDDs)
```python
rdd = sc.parallelize([(1, "a"), (2, "b"), (3, "c")])
partitioned = rdd.partitionBy(2, lambda x: x % 2)
```

---

## 2. ACTIONS (Eager Operations)

Actions trigger computation and return results to the driver or write to storage.

#### `collect()`
**Use**: Return all elements to driver as a list
```python
result = rdd.collect()  # [1, 2, 3, 4]
```
**⚠️ Warning**: Don't use on large RDDs - may cause out-of-memory errors

#### `count()`
**Use**: Count number of elements
```python
count = rdd.count()  # 4
```

#### `first()`
**Use**: Return first element
```python
first_elem = rdd.first()  # 1
```

#### `take(n)`
**Use**: Return first n elements
```python
first_three = rdd.take(3)  # [1, 2, 3]
```

#### `takeSample(withReplacement, num, [seed])`
**Use**: Return random sample of n elements
```python
sample = rdd.takeSample(False, 2, seed=42)  # [2, 4]
```

#### `takeOrdered(n, [ordering])`
**Use**: Return first n elements in natural order or custom ordering
```python
rdd = sc.parallelize([3, 1, 4, 2])
ordered = rdd.takeOrdered(2)  # [1, 2]
desc = rdd.takeOrdered(2, key=lambda x: -x)  # [4, 3]
```

#### `top(n, [ordering])`
**Use**: Return top n elements (descending order)
```python
top_two = rdd.top(2)  # [4, 3]
```

#### `reduce(func)`
**Use**: Aggregate elements using a reduce function
```python
rdd = sc.parallelize([1, 2, 3, 4])
sum_all = rdd.reduce(lambda x, y: x + y)  # 10
```

#### `fold(zeroValue, func)`
**Use**: Like reduce but with an initial zero value
```python
sum_with_init = rdd.fold(5, lambda x, y: x + y)  # 15 (5 + 1 + 2 + 3 + 4)
```

#### `aggregate(zeroValue, seqOp, combOp)`
**Use**: Aggregate with different types for accumulator and elements
```python
# Calculate sum and count
sum_count = rdd.aggregate(
    (0, 0),  # (sum, count)
    lambda acc, value: (acc[0] + value, acc[1] + 1),
    lambda acc1, acc2: (acc1[0] + acc2[0], acc1[1] + acc2[1])
)  # (10, 4)
```

#### `foreach(func)`
**Use**: Apply function to each element (side effects only, returns nothing)
```python
rdd.foreach(lambda x: print(x))  # Print each element
```

#### `foreachPartition(func)`
**Use**: Apply function to each partition (useful for batch operations)
```python
def save_partition(iterator):
    # Open database connection once per partition
    for record in iterator:
        # Save record
        pass
        
rdd.foreachPartition(save_partition)
```

#### `countByKey()`
**Use**: Count occurrences of each key (for key-value RDDs)
```python
rdd = sc.parallelize([("a", 1), ("b", 2), ("a", 3)])
counts = rdd.countByKey()  # {"a": 2, "b": 1}
```

#### `countByValue()`
**Use**: Count occurrences of each unique value
```python
rdd = sc.parallelize([1, 2, 2, 3, 3, 3])
value_counts = rdd.countByValue()  # {1: 1, 2: 2, 3: 3}
```

#### `saveAsTextFile(path)`
**Use**: Save RDD to text files
```python
rdd.saveAsTextFile("/tmp/output")
```

#### `saveAsSequenceFile(path)` (Java/Scala only)
**Use**: Save as Hadoop SequenceFile

#### `saveAsObjectFile(path)`
**Use**: Save using Java serialization
```python
rdd.saveAsObjectFile("/tmp/objects")
```

---

## 3. PERSISTENCE OPERATIONS

#### `cache()`
**Use**: Persist RDD in memory (default storage level)
```python
cached_rdd = rdd.cache()
```

#### `persist(storageLevel)`
**Use**: Persist with specific storage level
```python
from pyspark import StorageLevel
persisted = rdd.persist(StorageLevel.MEMORY_AND_DISK)
```

**Storage Levels**:
* `MEMORY_ONLY` - Store in memory only
* `MEMORY_AND_DISK` - Spill to disk if memory full
* `MEMORY_ONLY_SER` - Store as serialized objects
* `DISK_ONLY` - Store only on disk
* `OFF_HEAP` - Store in off-heap memory

#### `unpersist()`
**Use**: Remove RDD from cache
```python
rdd.unpersist()
```

---

## 4. UTILITY OPERATIONS

#### `getNumPartitions()`
**Use**: Get number of partitions
```python
num_parts = rdd.getNumPartitions()
```

#### `isEmpty()`
**Use**: Check if RDD is empty
```python
is_empty = rdd.isEmpty()
```

#### `toDebugString()`
**Use**: Get RDD lineage as string (for debugging)
```python
lineage = rdd.toDebugString()
```

---

## Quick Reference: When to Use What

| Task | Operation |
| --- | --- |
| Transform each element | `map()` |
| Filter data | `filter()` |
| Split and flatten | `flatMap()` |
| Aggregate by key | `reduceByKey()` |
| Group by key | `groupByKey()` (avoid if possible) |
| Join datasets | `join()`, `leftOuterJoin()`, etc. |
| Remove duplicates | `distinct()` |
| Sort by key | `sortByKey()` |
| Get all results | `collect()` (careful with size) |
| Count elements | `count()` |
| Get sample | `take()`, `takeSample()` |
| Aggregate all | `reduce()`, `aggregate()` |
| Save to file | `saveAsTextFile()` |
| Reuse RDD | `cache()`, `persist()` |

Summary of RDD Operations
1. TRANSFORMATIONS (Lazy - Build Execution Plan)

Narrow Transformations (no shuffle):

map() - Transform each element
filter() - Select matching elements
flatMap() - Map and flatten results
mapPartitions() - Process entire partitions efficiently
union() - Combine RDDs
distinct() - Remove duplicates
Wide Transformations (require shuffle):

reduceByKey() - Aggregate by key (preferred over groupByKey)
groupByKey() - Group values by key
sortByKey() - Sort by keys
join(), leftOuterJoin(), rightOuterJoin(), fullOuterJoin() - Join operations
cogroup() - Group from multiple RDDs
repartition() - Change partition count
aggregateByKey() - Complex aggregations
2. ACTIONS (Eager - Trigger Computation)

collect() - Return all elements to driver
count() - Count elements
first(), take(n) - Get first element(s)
reduce() - Aggregate all elements
foreach() - Apply side effects
saveAsTextFile() - Write to storage
countByKey(), countByValue() - Count statistics
3. PERSISTENCE

cache() - Store in memory
persist(level) - Custom storage level
unpersist() - Remove from cache
The cell includes detailed code examples, use cases, and warnings for each operation.



## What is DAG?

**DAG (Directed Acyclic Graph)** is Spark's execution model that represents the sequence of computations on RDDs/DataFrames. It's a graph of operations where:

* **Directed**: Data flows in one direction (from input to output)
* **Acyclic**: No cycles or loops - prevents infinite execution
* **Graph**: Nodes represent RDDs/DataFrames, edges represent transformations

When you submit a Spark job, the DAG Scheduler creates a logical execution plan by chaining all transformations together.

---

## Structure of DAG

### Components:

**1. Nodes (Vertices)**
* Represent RDDs or DataFrames at each stage
* Each node is the result of a transformation

**2. Edges (Arrows)**
* Represent transformations applied to data
* Show dependencies between RDDs

**3. Stages**
* Groups of transformations that can be executed together
* Separated by **shuffle boundaries** (wide transformations)

**4. Tasks**
* Smallest unit of work sent to executors
* One task per partition per stage

### Example DAG Structure:

```
TextFile RDD
    |
    v (map transformation)
Mapped RDD
    |
    v (filter transformation)
Filtered RDD
    |
    v (reduceByKey - SHUFFLE)
Reduced RDD
    |
    v (collect action)
   Result
```

---

## Types of DAG Operations

### 1. Narrow Transformations (No Shuffle)

**Characteristics**:
* Each input partition contributes to only one output partition
* No data movement across partitions
* Can be pipelined in a single stage
* Fast and efficient

**Examples**:
* `map()`, `filter()`, `flatMap()`
* `mapPartitions()`, `union()`
* `sample()`

**DAG Pattern**:
```
Partition 1 → Partition 1
Partition 2 → Partition 2
Partition 3 → Partition 3
(1:1 relationship)
```

### 2. Wide Transformations (Shuffle Required)

**Characteristics**:
* Data from multiple input partitions may contribute to one output partition
* Requires data shuffle across the cluster
* Creates stage boundaries in DAG
* Expensive operation (disk I/O, network transfer)

**Examples**:
* `reduceByKey()`, `groupByKey()`, `sortByKey()`
* `join()`, `cogroup()`, `repartition()`
* `distinct()`, `intersection()`

**DAG Pattern**:
```
Partition 1 ──┐
Partition 2 ──┼─→ Partition 1
Partition 3 ──┘
(M:N relationship)
```

---

## DAG Execution Workflow

### Step-by-Step Process:

**1. Job Submission**
* User calls an action (e.g., `collect()`, `count()`)
* Triggers DAG creation

**2. DAG Construction**
* DAG Scheduler builds logical plan
* Chains all transformations from source to action
* Creates RDD lineage graph

**3. Stage Division**
* Divides DAG into stages at shuffle boundaries
* Each wide transformation creates a new stage
* Stages are executed sequentially

**4. Task Creation**
* Each stage is divided into tasks
* Number of tasks = number of partitions
* Tasks are the smallest unit of work

**5. Task Scheduling**
* Task Scheduler assigns tasks to executors
* Tasks within a stage can run in parallel
* Data locality is considered

**6. Task Execution**
* Executors run tasks on partitions
* Results are either shuffled or returned

**7. Result Collection**
* Final results aggregated and returned to driver

### Visual Example:

```
Action (collect)
    |
    | Job
    |
    v
+-------------------+
| Stage 2           |  ← Shuffle boundary
| (reduceByKey)     |
+-------------------+
    ^
    | Shuffle
    |
+-------------------+
| Stage 1           |
| (map + filter)    |
+-------------------+
    |
    v
  Input Data
```

---

## DAG Optimization Techniques

### 1. Transformation Optimization

#### a) Prefer `reduceByKey()` over `groupByKey()`
**Why**: reduceByKey combines values locally before shuffling

```python
# ❌ BAD - shuffles all data
rdd.groupByKey().mapValues(sum)

# ✅ GOOD - reduces data before shuffle
rdd.reduceByKey(lambda x, y: x + y)
```

#### b) Filter Early
Reduce data volume before expensive operations

```python
# ✅ GOOD - filter before join
filtered_rdd = rdd.filter(lambda x: x[1] > 100)
result = filtered_rdd.join(other_rdd)
```

#### c) Use `coalesce()` instead of `repartition()` when reducing partitions
**Why**: coalesce avoids full shuffle when decreasing partitions

```python
# ❌ BAD - full shuffle
rdd.repartition(10)

# ✅ GOOD - no shuffle if reducing
rdd.coalesce(10)
```

### 2. Caching and Persistence

#### Cache frequently accessed RDDs

```python
# RDD used multiple times
cached_rdd = rdd.map(expensive_operation).cache()

result1 = cached_rdd.filter(condition1).count()
result2 = cached_rdd.filter(condition2).count()
# Only computes expensive_operation once
```

#### Choose appropriate storage level

```python
from pyspark import StorageLevel

# For large RDDs
rdd.persist(StorageLevel.MEMORY_AND_DISK)

# For memory-constrained scenarios
rdd.persist(StorageLevel.MEMORY_AND_DISK_SER)
```

### 3. Partition Optimization

#### a) Optimal Partition Count
**Rule of thumb**: 2-4 partitions per CPU core

```python
# Check current partitions
num_partitions = rdd.getNumPartitions()

# Optimize based on cluster size
optimal_partitions = num_cores * 3
rdd = rdd.repartition(optimal_partitions)
```

#### b) Custom Partitioning
For skewed data, use custom partitioners

```python
def custom_partitioner(key):
    # Custom logic to distribute data evenly
    return hash(key) % num_partitions

rdd.partitionBy(num_partitions, custom_partitioner)
```

### 4. Minimize Shuffles

#### a) Broadcast Small Tables
Avoid shuffle by broadcasting small datasets

```python
from pyspark.sql.functions import broadcast

# Instead of regular join (shuffle)
df1.join(df2, "key")

# Broadcast join (no shuffle for small table)
df1.join(broadcast(df2), "key")
```

#### b) Pre-partition Data

```python
# Partition both RDDs the same way before join
rdd1_partitioned = rdd1.partitionBy(num_partitions)
rdd2_partitioned = rdd2.partitionBy(num_partitions)

# Join without additional shuffle
joined = rdd1_partitioned.join(rdd2_partitioned)
```

### 5. Pipelining Narrow Transformations

Chain multiple narrow transformations - they execute in a single stage

```python
# All narrow transformations - single stage
result = rdd \
    .map(lambda x: x * 2) \
    .filter(lambda x: x > 10) \
    .map(lambda x: (x, 1)) \
    .reduceByKey(lambda x, y: x + y)  # Stage boundary here
```

### 6. Avoid Wide Dependencies When Possible

```python
# ❌ BAD - multiple shuffles
rdd.distinct().sortBy(lambda x: x).groupByKey()

# ✅ GOOD - combine operations
rdd.aggregateByKey(...)  # Single shuffle
```

### 7. Use DataFrame/Dataset API

Catalyst optimizer automatically optimizes logical plans

```python
# DataFrame API with automatic optimization
df.filter(col("value") > 100) \
  .groupBy("key") \
  .agg(sum("value")) \
  .orderBy("key")

# Catalyst optimizes:
# - Predicate pushdown
# - Projection pruning
# - Constant folding
```

### 8. Speculative Execution

Enable for handling straggler tasks

```python
spark.conf.set("spark.speculation", "true")
spark.conf.set("spark.speculation.multiplier", "1.5")
```

### 9. Dynamic Partition Pruning

For joins with filters, automatically skip partitions

```python
spark.conf.set("spark.sql.optimizer.dynamicPartitionPruning.enabled", "true")
```

### 10. Adaptive Query Execution (AQE)

Dynamically optimize execution plan at runtime

```python
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
```

---

## DAG Visualization

### View DAG in Spark UI

**Spark UI → Jobs → [Job ID] → DAG Visualization**

Shows:
* Visual representation of stages
* Dependencies between stages
* Shuffle boundaries
* Task execution metrics

### Programmatic DAG Inspection

```python
# View RDD lineage
print(rdd.toDebugString().decode())

# Output shows dependencies:
# (2) PythonRDD[3] at RDD at PythonRDD.scala:53
#  |  MapPartitionsRDD[2] at mapPartitions at PythonRDD.scala:133
#  |  ShuffledRDD[1] at partitionBy at NativeMethodAccessorImpl.java:0
#  +-(2) PairwiseRDD[0] at reduceByKey at <stdin>:1
```

---

## Common DAG Patterns

### 1. Linear Pipeline
```
Input → map → filter → map → collect
(Single stage - all narrow transformations)
```

### 2. Shuffle Pipeline
```
Stage 1: Input → map → filter
    ↓ (shuffle)
Stage 2: reduceByKey → map → collect
```

### 3. Join Pattern
```
Stage 1: RDD1 → map → filter
Stage 2: RDD2 → map
    ↓ (shuffle for join)
Stage 3: join → map → collect
```

### 4. Multi-Output Pattern
```
        Input → map → filter → cache
           ↓              ↓
    Action 1          Action 2
(Cache prevents recomputation)
```

---

## Best Practices for DAG Optimization

1. **Minimize Shuffles**: Most expensive operation
2. **Cache Strategically**: Cache RDDs used multiple times
3. **Filter Early**: Reduce data volume before expensive ops
4. **Use Appropriate Transformations**: `reduceByKey` > `groupByKey`
5. **Partition Wisely**: Balance parallelism and overhead
6. **Leverage Broadcast**: For small lookup tables
7. **Monitor Spark UI**: Identify bottlenecks
8. **Use DataFrame API**: Automatic optimization
9. **Avoid Collect on Large Data**: Use `take()` or sampling
10. **Pipeline Narrow Transformations**: Execute in single stage

---

## Key Takeaways

* **DAG enables lazy evaluation**: Optimize before execution
* **Stages defined by shuffles**: Wide transformations create boundaries
* **Narrow transformations are fast**: No data movement
* **Wide transformations are expensive**: Network and disk I/O
* **Catalyst optimizer**: DataFrames get automatic optimization
* **Caching prevents recomputation**: Essential for iterative algorithms
* **Spark UI is your friend**: Visualize and debug DAG execution

Quick Summary
DAG (Directed Acyclic Graph) is Spark's execution model - a graph representing the sequence of computations where data flows in one direction without cycles.

Structure Components:

Nodes - RDDs/DataFrames at each transformation
Edges - Transformations connecting nodes
Stages - Groups of operations separated by shuffle boundaries
Tasks - Smallest unit of work (one per partition per stage)
Two Operation Types:

Narrow Transformations (fast, no shuffle):

1:1 partition mapping - map, filter, flatMap
Pipelined in single stage
Wide Transformations (expensive, require shuffle):

M:N partition mapping - reduceByKey, join, groupByKey
Create stage boundaries
Top 10 Optimization Techniques:

Use reduceByKey() over groupByKey()
Filter early to reduce data volume
Cache frequently accessed RDDs
Optimize partition count (2-4 per CPU core)
Minimize shuffles
Broadcast small tables in joins
Use coalesce() instead of repartition() when reducing
Pipeline narrow transformations
Use DataFrame API for Catalyst optimization
Enable Adaptive Query Execution (AQE)
The cell includes detailed examples, DAG patterns, execution workflow, and visualization techniques.
